# Document Question Answering with RAG

A simple Retrieval-Augmented Generation system built on the Open RAG Benchmark dataset from Hugging Face. We load research papers from the dataset, split them into chunks, store them as vectors, and answer questions by retrieving the most relevant chunks and passing them to a language model.


## Install

In [1]:
# !pip install huggingface_hub sentence-transformers faiss-cpu transformers
# !pip install ollama
# !pip uninstall -y torch
# !pip install torch --index-url https://download.pytorch.org/whl/cu121

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


## Imports

In [3]:
import json
import numpy as np
from huggingface_hub import hf_hub_download, list_repo_files
from sentence_transformers import SentenceTransformer
import faiss
from transformers import pipeline

print("Libraries loaded")

C:\Users\Alpha\Desktop\Celebal_Technology\.celebal\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded


## Find the Dataset Files

The dataset stores each paper as a separate JSON file, plus three metadata files: the questions (queries), the relevance labels that link each question to its paper (qrels), and the reference answers. We locate these first.

In [4]:
REPO = "vectara/open_ragbench"

files = list_repo_files(REPO, repo_type="dataset")

queries_path = [f for f in files if f.endswith("queries.json")][0]
qrels_path   = [f for f in files if f.endswith("qrels.json")][0]
answers_path = [f for f in files if f.endswith("answers.json")][0]
corpus_files = [f for f in files if "/corpus/" in f and f.endswith(".json")]

print("queries file:", queries_path)
print("qrels file  :", qrels_path)
print("answers file:", answers_path)
print("papers in corpus:", len(corpus_files))

queries file: pdf/arxiv/queries.json
qrels file  : pdf/arxiv/qrels.json
answers file: pdf/arxiv/answers.json
papers in corpus: 1000


## Load the Metadata

In [5]:
def load_json(filename):
    local = hf_hub_download(REPO, filename=filename, repo_type="dataset")
    with open(local, "r", encoding="utf-8") as f:
        return json.load(f)

queries = load_json(queries_path)
qrels   = load_json(qrels_path)
answers = load_json(answers_path)

print("Total questions:", len(queries))
print("Total answers  :", len(answers))

Total questions: 3045
Total answers  : 3045


## Pick a Few Papers

Loading all 1000 papers would be slow, so we pick a small number that have questions attached to them. We build a lookup from paper id to its file path, then choose papers referenced in qrels.

In [6]:
num_papers = 5
corpus_lookup = {}
for path in corpus_files:
    pid = path.split("/")[-1].replace(".json", "")
    corpus_lookup[pid] = path
wanted_docs = []
for q_id, rel in qrels.items():
    doc_id = rel["doc_id"]
    if doc_id in corpus_lookup and doc_id not in wanted_docs:
        wanted_docs.append(doc_id)
    if len(wanted_docs) >= num_papers:
        break

print("Papers we will load:", wanted_docs)

Papers we will load: ['2410.14077v2', '2404.00822v2', '2410.07168v2', '2401.07294v4', '2411.14884v3']


## Download and Read the Papers

Each paper has an abstract and several sections. We join them into one block of text per paper.

In [7]:
documents = [] 
for pid in wanted_docs:
    paper = load_json(corpus_lookup[pid])
    text = paper.get("abstract", "")
    for sec in paper.get("sections", []):
        text += "\n" + sec.get("text", "")
    documents.append((pid, text))
    print(pid, "->", len(text), "characters")

2410.14077v2 -> 47872 characters
2404.00822v2 -> 80309 characters
2410.07168v2 -> 91948 characters
2401.07294v4 -> 73990 characters
2411.14884v3 -> 37964 characters


## Split into Chunks

We break each paper into smaller overlapping chunks and remember which paper each chunk came from.

In [8]:
def chunk_text(text, chunk_size=200, overlap=40):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        chunks.append(" ".join(words[start:start + chunk_size]))
        start += chunk_size - overlap
    return chunks

all_chunks = []
chunk_sources = []
for pid, text in documents:
    for ch in chunk_text(text):
        all_chunks.append(ch)
        chunk_sources.append(pid)

print("Total chunks:", len(all_chunks))

Total chunks: 289


## Create Embeddings and Build the Vector Store

Each chunk becomes a vector. We store the vectors in a FAISS index and normalize them so similarity is measured by cosine.

In [9]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embed_model.encode(all_chunks, convert_to_numpy=True, show_progress_bar=True)
faiss.normalize_L2(chunk_embeddings)
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

print("Vectors stored:", index.ntotal)

Batches: 100%|██████████| 10/10 [00:00<00:00, 25.38it/s]

Vectors stored: 289


## Retrieval

In [10]:
def retrieve(query, k=3):
    qv = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(qv)
    scores, idx = index.search(qv, k)
    return [all_chunks[i] for i in idx[0]]

## Load the Generator

Qwen is a small instruction-following model. It reads the question and the retrieved chunks and writes an answer.

In [11]:
import torch
torch.set_num_threads(4)
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
gen_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device)
print("Generator ready ->", MODEL_NAME, "on", device)

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 819.62it/s]


Generator ready -> Qwen/Qwen2.5-3B-Instruct on cuda


## Answer Function

Retrieve the relevant chunks, put them in a prompt as context, and generate the answer.

In [12]:
def answer_question(query, k=3):
    context = "\n".join(retrieve(query, k))
    messages = [
        {"role": "system", "content": "Answer the question using only the context provided. If the answer is not in the context, say so clearly."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]
    text = gen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = gen_tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(device)
    output = gen_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return gen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## Test with Real Questions from the Dataset

We pull questions that belong to the papers we loaded, generate answers, and show the dataset's reference answer next to ours for comparison.

In [13]:
print(f"Using model -> {MODEL_NAME} on {device}")
print("Ask a question about the loaded papers. Type 'quit' to stop.\n")
while True:
    user_query = input("Your question: ").strip()
    if user_query.lower() in ("quit", "exit", ""):
        print("Done.")
        break
    print("Answer:", answer_question(user_query))
    print("-" * 60)

Using model -> Qwen/Qwen2.5-3B-Instruct on cuda
Ask a question about the loaded papers. Type 'quit' to stop.



Your question:  In what ways can this methodological approach impact future curriculum design across disciplines?


Answer: The context provided does not contain specific information about how this methodological approach impacts future curriculum design across disciplines. It primarily focuses on various methodologies used in statistical and educational research, such as Monte Carlo simulations, mixed-effects models, and hierarchical linear models. Without additional details linking these methodologies directly to curriculum design, it's not possible to determine their potential impact on future curriculum design.
------------------------------------------------------------


Your question:  In what areas do syllabic embeddings show potential for improvement based on current research findings?


Answer: Based on the current research findings, syllabic embeddings show potential for improvement in pitch representation, where they result in a flattened speech generation pattern compared to non-quantized models. Additionally, HuBERT units with finer clustering (100 or more clusters) provide slightly better articulatory reconstruction and slightly better intelligibility, though they require significantly more tokens per second.
------------------------------------------------------------


...

Answer: Changes in the effective microbial death rate influence parameters like α and β such that both α and β decrease with increasing heterogeneity in the effective microbial death rate, and both α and β increase when increasing the expectation of the effective microbial death rate.
------------------------------------------------------------


Your question:  What are the challenges in estimating output impedance in inverter-based grids?


Answer: The challenges in estimating output impedance in inverter-based grids include:

1. Lack of access to global measurements or networkwide data, making it difficult to estimate the effective grid voltage.
2. Measured signals often lack the necessary persistence of excitation, which is crucial for accurate impedance estimation.
3. Only local output voltage and current are measurable, while both line impedance and grid voltage influence these measurements, making it essential to distinguish between their effects.
4. It is often impractical or not allowed to alter the power system to assist in impedance estimation.
5. Estimation must be non-invasive, relying solely on locally measured signals without injecting real power disturbances into the system.
------------------------------------------------------------


Your question:  


Done.


## Conclusion

This is a complete RAG system running on the Open RAG Benchmark dataset. It downloads real research papers, chunks them, stores them as vectors in FAISS, retrieves the most relevant chunks for a question, and generates a grounded answer. 